TODO:

1. Confirm all E2E scans are exported
2. Load all E2E scans
3. Run preprocessing on each scan
   - extract volume
   - extract RPE/BM
   - flatten using RPE
   - extract below-RPE ROI

4. Compute QC table
   - missing layers
   - failed flattening
   - ROI dimensions
   - intensity summaries/histograms
   - scan quality
   - acquisition parameters

5. Save processed outputs
   - flattened volume or ROI volume
   - metadata/QC CSV

6. Create clinician review file
   - ID
   - representative B-scans to review
   - barcode status: absent/present/unsure
   - optional notes

7. After clinician labeling, merge labels with QC table
   - id
   - barcode status
   - number of B-scans
   - flatten ok
   - ROI extracted
   - dimensions

Final File Structure:
```
id, barcode_status, label_confidence, notes, n_bscans, missing_layers, flatten_ok, roi_extracted, roi_shape, scan_quality
```

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from barcode.preprocessing import run_preprocessing_pipeline

DATA_DIR = PROJECT_ROOT / "data" / "heyex"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [2]:
qc_df, label_df = run_preprocessing_pipeline(
    data_dir=DATA_DIR,
    processed_dir=PROCESSED_DIR,
    n_patients=10,
    save_flattened=False,
)

Found 54 E2E files.
[1/54] 001 | 001001.E2E
[2/54] 001 | 001002.E2E
[3/54] 001 | 001003.E2E
[4/54] 001 | 001004.E2E
[5/54] 001 | 001005.E2E
[6/54] 001 | 001006.E2E
[7/54] 002 | 002001.E2E
[8/54] 002 | 002002.E2E
[9/54] 002 | 002003.E2E
[10/54] 003 | 003001.E2E
[11/54] 003 | 003002.E2E
[12/54] 003 | 003003.E2E
[13/54] 003 | 003004.E2E
[14/54] 003 | 003005.E2E
[15/54] 003 | 003006.E2E
[16/54] 004 | 004001.E2E
[17/54] 004 | 004002.E2E
[18/54] 004 | 004003.E2E
[19/54] 004 | 004004.E2E
[20/54] 004 | 004005.E2E
[21/54] 004 | 004006.E2E
[22/54] 005 | 005001.E2E
[23/54] 005 | 005002.E2E
[24/54] 005 | 005003.E2E
[25/54] 005 | 005004.E2E
[26/54] 005 | 005005.E2E
[27/54] 005 | 005006.E2E
[28/54] 006 | 006001.E2E
[29/54] 006 | 006002.E2E
[30/54] 006 | 006003.E2E
[31/54] 006 | 006004.E2E
[32/54] 006 | 006005.E2E
[33/54] 006 | 006006.E2E
[34/54] 007 | 007001.E2E
[35/54] 007 | 007002.E2E
[36/54] 007 | 007003.E2E
[37/54] 007 | 007004.E2E
[38/54] 007 | 007005.E2E
[39/54] 008 | 008001.E2E
[40/54] 008 | 

In [3]:
qc_df.head()

,patient_id,file_name,e2e_path,loaded_ok,preprocess_ok,roi_saved,flattened_saved,error,shape,n_bscans,...,quality,n_bscans_meta,scan_pattern,start_x,start_y,end_x,end_y,roi_path,roi_shape,target_y
0,001,001001.E2E,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,True,True,True,False,None,"(97, 496, 512)",97,...,None,None,None,None,None,None,None,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",157
1,001,001002.E2E,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,True,True,True,False,None,"(97, 496, 512)",97,...,None,None,None,None,None,None,None,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",187
2,001,001003.E2E,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,True,True,True,False,None,"(97, 496, 512)",97,...,None,None,None,None,None,None,None,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",155
3,001,001004.E2E,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,True,True,True,False,None,"(97, 496, 512)",97,...,None,None,None,None,None,None,None,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",181
4,001,001005.E2E,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,True,True,True,False,None,"(97, 496, 512)",97,...,None,None,None,None,None,None,None,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",187


In [4]:
label_df.head()

,patient_id,file_name,n_bscans,height,width,roi_path,roi_shape,has_rpe,has_bm,target_y,barcode_volume_status,first_positive_bscan,last_positive_bscan,label_confidence,clinician_notes
0,001,001001.E2E,97,496,512,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",True,True,157,,,,,
1,001,001002.E2E,97,496,512,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",True,True,187,,,,,
2,001,001003.E2E,97,496,512,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",True,True,155,,,,,
3,001,001004.E2E,97,496,512,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",True,True,181,,,,,
4,001,001005.E2E,97,496,512,c:\Users\u0667281\Desktop\JMA\barcoding-amd\da...,"(97, 155, 512)",True,True,187,,,,,


In [ ]:
# For a full run
qc_df, label_df = run_preprocessing_pipeline(
    data_dir=DATA_DIR,
    processed_dir=PROCESSED_DIR,
    n_patients=None,
    save_flattened=False,
)